In [4]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# ====================== LOAD EXCEL ======================
df = pd.read_excel("repair_dataset.xlsx")

# Basic cleaning
df = df.dropna(subset=['Date_In', 'Date_Out'])
df['Date_In'] = pd.to_datetime(df['Date_In'])
df['Date_Out'] = pd.to_datetime(df['Date_Out'])
df['Duration_Days'] = (df['Date_Out'] - df['Date_In']).dt.days

df['Year'] = df['Date_In'].dt.year
df['Month'] = df['Date_In'].dt.month
df['Day'] = df['Date_In'].dt.day

# Text Cleaning for NLP
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Fault_Clean'] = df['Fault_Description'].apply(clean_text)

print("✅ Data Loaded Successfully!")
print("Shape:", df.shape)
df.head()

✅ Data Loaded Successfully!
Shape: (2520, 15)


,Job_ID,Date_In,Device_Type,Item_Model,Fault_Description,Technician,Repair_Path,Warranty,Solution,Date_Out,Duration_Days,Year,Month,Day,Fault_Clean
0,4572.0,2023-01-01,Desktop PC,Core 2 Duo PC,No Power,Nimales,In-House,No,Power Supply Replaced,2023-01-02,1,2023,1,1,no power
1,4573.0,2023-01-01,Desktop PC,Core i3 Desktop,Display Flickering,Pradeep,In-House,No,RAM Replacement,2023-01-01,0,2023,1,1,display flickering
2,4574.0,2023-01-02,Desktop PC,Asus H410 PC,No Power,Nimales,In-House,No,Motherboard Service,2023-01-05,3,2023,1,2,no power
3,4575.0,2023-01-02,UPS,Prolink UPS,No Backup,Nimales,In-House,No,Battery Replacement,2023-01-02,0,2023,1,2,no backup
4,4576.0,2023-01-02,UPS,DCP UPS,No Power,Nimales,In-House,No,Fuse Replacement,2023-01-03,1,2023,1,2,no power


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import joblib

# ====================== 1. CLEAN DATA ======================
df = df.dropna(subset=['Fault_Description', 'Duration_Days']).reset_index(drop=True)
print("Final dataset shape:", df.shape)

# ====================== 2. STRONG NLP ======================
tfidf = TfidfVectorizer(
    max_features=350, 
    stop_words='english', 
    ngram_range=(1,3),
    min_df=2,
    max_df=0.95
)
tfidf_matrix = tfidf.fit_transform(df['Fault_Clean'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f"fault_{i}" for i in range(tfidf_matrix.shape[1])])

# ====================== 3. ENCODE CATEGORICAL ======================
categorical_cols = ['Device_Type', 'Item_Model', 'Technician', 'Repair_Path', 'Warranty', 'Solution']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# ====================== 4. FINAL FEATURES ======================
X = pd.concat([df[categorical_cols + ['Year', 'Month', 'Day']], tfidf_df], axis=1)
y = df['Duration_Days']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ====================== 5. BEST MODEL ======================
model = RandomForestRegressor(
    n_estimators=1200,
    max_depth=45,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\n🏆 BEST MODEL TRAINING COMPLETED!")
print("R2 Score:", round(r2_score(y_test, y_pred), 4))
print("MAE (days):", round(mean_absolute_error(y_test, y_pred), 4))

# Feature Importance
importances = pd.Series(model.feature_importances_, index=X.columns)
print("\n🔝 Top 15 Most Important Features:")
print(importances.nlargest(15))

# Save Model
joblib.dump(model, "best_model_nlp.pkl")
joblib.dump(encoders, "encoders.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
print("\n✅ Best Model Saved Successfully!")

Final dataset shape: (2520, 15)

🏆 BEST MODEL TRAINING COMPLETED!
R2 Score: 0.8945
MAE (days): 1.1567

🔝 Top 15 Most Important Features:
Repair_Path    0.179769
fault_41       0.073892
fault_40       0.068399
fault_118      0.064679
Warranty       0.059023
Solution       0.042483
fault_133      0.032282
fault_138      0.031912
fault_17       0.031372
fault_18       0.030352
fault_73       0.028310
fault_20       0.027883
fault_25       0.023496
fault_67       0.023016
fault_54       0.022731
dtype: float64

✅ Best Model Saved Successfully!


In [35]:
from datetime import datetime, timedelta
import joblib
import re
import pandas as pd

# ====================== LOAD MODEL ======================
model = joblib.load("best_model_nlp.pkl")
encoders = joblib.load("encoders.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")

categorical_cols = ['Device_Type', 'Item_Model', 'Technician', 
                    'Repair_Path', 'Warranty', 'Solution']

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ====================== PREDICTION FUNCTION ======================
def predict_repair(new_job):
    date_in = datetime.strptime(new_job['Date_In'], "%Y-%m-%d")
    
    # Categorical Features
    features = {}
    for col in categorical_cols:
        le = encoders[col]
        val = new_job.get(col, "Unknown")
        features[col] = le.transform([val])[0] if val in le.classes_ else -1
    
    # NLP - Fault Description
    fault_clean = clean_text(new_job['Fault_Description'])
    fault_vector = tfidf.transform([fault_clean]).toarray()
    fault_df = pd.DataFrame(fault_vector, columns=[f"fault_{i}" for i in range(fault_vector.shape[1])])
    
    # Date Features
    features['Year'] = date_in.year
    features['Month'] = date_in.month
    features['Day'] = date_in.day
    
    # Create input
    X_new = pd.DataFrame([features])
    X_new = pd.concat([X_new, fault_df], axis=1)
    
    # Pure ML Prediction
    raw_pred = model.predict(X_new)[0]
    pred_days = round(raw_pred)
    
    print("🔍 Debug Info:")
    print(f"Fault Description: {new_job['Fault_Description']}")
    print(f"Repair Path: {new_job['Repair_Path']}")
    print(f"Solution: {new_job['Solution']}")
    print(f"Raw Prediction: {raw_pred:.2f} days")
    
    completion_date = date_in + timedelta(days=pred_days)
    
    return pred_days, completion_date.strftime("%Y-%m-%d")


# ====================== TEST CASE ======================
new_job = {
    "Date_In": "2026-03-07",
    "Device_Type": "Laptop",
    "Item_Model": "HP Pavilion",
    "Fault_Description": "No Power",
    "Technician": "Nimales",
    "Repair_Path": "Agent",
    "Warranty": "No",
    "Solution": "Logic Board Repair"
}

duration, completion = predict_repair(new_job)

print(f"\n🔧 Final Prediction: {duration} days")
print(f"📅 Expected Completion Date: {completion}")

🔍 Debug Info:
Fault Description: No Power
Repair Path: Agent
Solution: Logic Board Repair
Raw Prediction: 7.04 days

🔧 Final Prediction: 7 days
📅 Expected Completion Date: 2026-03-14
